# Taller Capítulo 4 — Ruben Feng
## Actividad 3. Medición de tiempo y complejidad de un algoritmo en Python

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('notas_completas_capitulo4.csv')
print(df.shape)
df.head()

### 3.1 Promedio por grupo con el dataset real (180 filas)

In [ ]:
def promedio_for(data):
    """Promedio por grupo con un ciclo for manual en Python puro: O(n) pero interpretado fila a fila."""
    suma, cuenta = {}, {}
    for grupo, nota in zip(data['Grupo'].tolist(), data['Nota'].tolist()):
        suma[grupo] = suma.get(grupo, 0) + nota
        cuenta[grupo] = cuenta.get(grupo, 0) + 1
    return {g: suma[g] / cuenta[g] for g in suma}

def medir(func, *args, repeticiones=5):
    """Devuelve el resultado y el mejor tiempo (s) de varias repeticiones, para reducir ruido."""
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.time()
        res = func(*args)
        tiempos.append(time.time() - inicio)
    return res, min(tiempos)

groupby_mean = lambda data: data.groupby('Grupo')['Nota'].mean()

prom_real, t_gb_real = medir(groupby_mean, df)
prom_for_real, t_for_real = medir(promedio_for, df)
print(prom_real)
print(f'groupby (180 filas): {t_gb_real:.6f} s')
print(f'for     (180 filas): {t_for_real:.6f} s')

# auto-chequeo: ambos métodos deben dar el mismo resultado
assert all(abs(prom_for_real[g] - prom_real[g]) < 1e-9 for g in prom_real.index)

### 3.2 Dataset sintético grande (200,000 filas, misma estructura)

In [ ]:
N = 200_000
rng = np.random.default_rng(42)
grupos = rng.choice(df['Grupo'].unique(), N)
df_grande = pd.DataFrame({
    'ID_Estudiante': [f'{g}-{i:06d}' for i, g in enumerate(grupos)],
    'Grupo': grupos,
    'Laboratorio': rng.choice(df['Laboratorio'].unique(), N),
    'Nota': np.clip(rng.normal(df['Nota'].mean(), df['Nota'].std(), N), 0, 100).round(1),
})
print(df_grande.shape)
df_grande.head()

### 3.3 Misma agregación: ciclo `for` manual vs `groupby().mean()`

In [ ]:
prom_for_grande, t_for_grande = medir(promedio_for, df_grande)
prom_gb_grande, t_gb_grande = medir(groupby_mean, df_grande)

# for con iterrows: la forma "ingenua" más común de recorrer un DataFrame fila a fila
def promedio_iterrows(data):
    suma, cuenta = {}, {}
    for _, fila in data.iterrows():
        suma[fila['Grupo']] = suma.get(fila['Grupo'], 0) + fila['Nota']
        cuenta[fila['Grupo']] = cuenta.get(fila['Grupo'], 0) + 1
    return {g: suma[g] / cuenta[g] for g in suma}

prom_it_grande, t_it_grande = medir(promedio_iterrows, df_grande, repeticiones=1)

print(f'for (zip)      : {t_for_grande:.4f} s')
print(f'for (iterrows) : {t_it_grande:.4f} s')
print(f'groupby().mean : {t_gb_grande:.4f} s')
assert all(abs(prom_for_grande[g] - prom_gb_grande[g]) < 1e-6 for g in prom_gb_grande.index)
assert all(abs(prom_it_grande[g] - prom_gb_grande[g]) < 1e-6 for g in prom_gb_grande.index)

### 3.4 Comparación de tiempos

In [ ]:
comparacion = pd.DataFrame({
    'Real (180 filas)': [t_for_real, t_gb_real],
    'Sintético (200,000 filas)': [t_for_grande, t_gb_grande],
}, index=['for manual', 'groupby().mean()'])
comparacion['Factor de crecimiento'] = comparacion['Sintético (200,000 filas)'] / comparacion['Real (180 filas)']
print(f'Las filas crecieron x{N / len(df):,.0f}')
print(f'groupby es x{t_for_grande / t_gb_grande:,.1f} más rápido que el for (zip) y x{t_it_grande / t_gb_grande:,.0f} más rápido que iterrows con 200,000 filas')
comparacion

### Conclusión Actividad 3

Con el dataset real (180 filas) ambas implementaciones terminan en fracciones de milisegundo; incluso el ciclo `for` puede igualar o superar a `groupby` porque el costo fijo de pandas (crear índices, preparar el objeto agrupado) domina sobre el poco trabajo real. **Con datos pequeños, medir el tiempo no permite distinguir un algoritmo correcto de uno equivocado.**

Al escalar a 200,000 filas (≈1,100 veces más datos) la diferencia aparece: ambos son O(n) en teoría, pero el `for` manual ejecuta cada iteración en el intérprete de Python, mientras que `groupby().mean()` hace el mismo recorrido en código C vectorizado. El `for` con `iterrows` (la forma ingenua típica) es órdenes de magnitud más lento, porque además construye un objeto `Series` por cada fila. El tiempo del `for` crece casi proporcional al número de filas, mientras que `groupby` crece mucho menos gracias a que su costo fijo se amortiza.

Esto ilustra la idea de **"algoritmo equivocado"** del capítulo: una solución puede ser correcta (los `assert` confirman que todos los métodos dan el mismo promedio) y aun así ser inadecuada porque no escala al volumen real que la visualización necesita procesar de forma interactiva. Por eso el nivel de algoritmo se valida de dos formas complementarias: **análisis de complejidad** (anticipa cómo crecerá el costo con *n*) y **medición del tiempo del sistema** con datos de tamaño realista (revela constantes ocultas, como el overhead del intérprete, que la notación O no muestra). Validar solo con el dataset pequeño habría ocultado el problema.

---
## Actividad 4. Prototipo rápido para validar la abstracción antes de programar

In [ ]:
# SITUACIÓN DE DOMINIO:
#   Usuarios: docente y coordinación del curso Bases de Datos II.
#   Pregunta: ¿qué grupo (GS123, GS124, GS126) tiene más dificultad a lo largo de los
#   cinco laboratorios, y en qué laboratorio(s) se acentúa, para decidir dónde reforzar?
#
# ABSTRACCIÓN DE DATOS:
#   Tabla de 180 ítems (estudiante x laboratorio). Atributos:
#     - Grupo: categórico (3 niveles)
#     - Laboratorio: ordinal (Lab1 < ... < Lab5, secuencia en el tiempo)
#     - Nota: cuantitativo (0-100)
#   Dato derivado: promedio de Nota por (Grupo, Laboratorio) -> tabla 3 x 5.
#
# ABSTRACCIÓN DE TAREA:
#   Comparar (compare) los valores entre grupos e identificar (identify) extremos:
#   el grupo con menor desempeño; secundariamente, descubrir tendencias entre laboratorios.

tabla = df.groupby(['Grupo', 'Laboratorio'])['Nota'].mean().unstack()
tabla.round(1)

### 4.1 Prototipo de baja fidelidad (líneas básicas, sin formato)

In [ ]:
for grupo in tabla.index:
    plt.plot(tabla.columns, tabla.loc[grupo], label=grupo)
plt.legend()
plt.show()

### 4.2 Retroalimentación del compañero

Se mostró el prototipo a un compañero sin dar explicaciones, con la pregunta: *¿Qué grupo tiene más dificultad?*

- **Compañero:** _(nombre)_
- **Respuesta:** _(escribir la respuesta textual del compañero)_
- **¿Identificó correctamente al grupo con más dificultad (GS124)?** _(Sí / No)_
- **Dudas o comentarios que tuvo:** _(p. ej., no sabía qué representaba el eje Y, no sabía que eran promedios)_

### 4.3 Decisión sobre la abstracción

En el prototipo la línea de **GS124** queda claramente por debajo de las otras dos en los cinco laboratorios (≈67–76 frente a ≈87–95), así que la tarea principal — *identificar el grupo con más dificultad* — se resuelve a simple vista. La abstracción de datos (promedio por grupo y laboratorio, tabla 3×5) y la de tarea (comparar grupos / identificar el mínimo) **tienen sentido y se mantienen**.

Ajustes que surgen del prototipo, antes de pasar a un idioma más elaborado:
1. Faltan título y etiquetas de ejes (qué es el valor: *nota promedio*, escala 0–100).
2. La línea sugiere continuidad entre laboratorios; como `Laboratorio` es ordinal y la tarea es comparar celdas (grupo × laboratorio), un **mapa de calor** con los valores anotados codifica mejor la tabla 3×5 y permite ver también en qué laboratorio se acentúa la dificultad (Lab3 para GS124).
3. Usar una escala de color secuencial y mostrar el número en cada celda para no depender solo del color.

_(Si el compañero no identificó a GS124, la decisión sería la misma pero con más peso en el punto 1: el problema sería de codificación/etiquetado, no de la abstracción.)_

In [ ]:
# Versión refinada tras validar la abstracción: mapa de calor con seaborn
plt.figure(figsize=(7, 3))
sns.heatmap(tabla, annot=True, fmt='.1f', cmap='YlOrRd_r', vmin=60, vmax=100,
            cbar_kws={'label': 'Nota promedio'})
plt.title('Nota promedio por grupo y laboratorio — Bases de Datos II')
plt.xlabel('Laboratorio')
plt.ylabel('Grupo')
plt.tight_layout()
plt.show()